# PDF Keyword + Appeal-Scope Harvester (ET Corpus → _Matches)

This script scans a large folder of ET-case PDFs and builds a **targeted "matches" library** for manual review and downstream analysis.

---

## Purpose

Turn a large, unstructured PDF corpus into a structured shortlist of potentially relevant cases using deterministic keyword and regex filtering.

This is a **high-speed recall harvester**, not a semantic or LLM-based precision layer.

---

## Step 1 — Crawl and Pre-Filter PDFs

* Recursively finds all `*.pdf` files under `INPUT_ROOT`.
* Rejects small documents (`pages < MIN_PAGES`).
* Extracts text from the first `TEXT_PAGES_TO_SCAN` pages (cheap front-scan).

This keeps processing fast while avoiding irrelevant small files.

---

## Step 2 — Two-Tier Matching Logic

### Gate Condition (Mandatory)

Every PDF must contain **all phrases in `NEEDLES_ALL`**.

If this fails → the document is ignored.

### Keep Condition (At Least One Required)

After passing the gate, the PDF must satisfy **at least one** of the following:

* Contain any phrase from `NEEDLES_ANY` (simple substring match), OR
* Match `APPEAL_SCOPE_REGEX` (detects appeal-scope limitation language such as:

  * "not raised in the appeal"
  * "outside the scope of the appeal"
  * "declined to consider"
  * "failed to engage with"
  * etc.)

Only documents passing Gate + Keep are retained.

---

## Step 3 — Parallel Scanning

* Uses `ProcessPoolExecutor` with up to `MAX_WORKERS` processes.
* Each PDF is scanned independently.

For matched files, metadata collected includes:

* `path`
* `pages`
* `size_mb`
* `mtime`
* `hit_all`
* `hit_any`
* `appeal_scope_hit`
* `appeal_scope_match` (small snippet of matched phrase)

---

## Step 4 — Structured Copy to Matches Folder

All matched PDFs are copied into `MATCHES_ROOT`.

### Folder Structure

* One folder per `NEEDLES_ANY` term (slugified)
* One dedicated folder `_APPEAL_SCOPE_REGEX` for all regex hits

If `PRESERVE_STRUCTURE = True`, original directory hierarchy is preserved under each subfolder.

This prevents filename collisions and preserves provenance.

---

## Step 5 — Master Index CSV

Exactly one CSV is written to:

`MATCHES_ROOT/_matches_index.csv`

The CSV contains:

* File metadata
* Raw hit indicators
* Boolean columns per NEEDLES_ANY (e.g., `has__predetermination`)
* Boolean column for regex hits (`has__appeal_scope_regex`)
* Root path reference

---

## Output Artifacts

1. Curated PDF library under `MATCHES_ROOT/`
2. Single master index CSV for analysis

---

## Architectural Position

This module is a **deterministic recall harvester**.

It does not perform:

* Semantic analysis
* LLM reasoning
* Paragraph-level anchoring

Its role is to reduce a massive PDF corpus into a manageable, topic-filtered working set for deeper analysis (e.g., Moltie or WBerious precision layers).

---

## Guiding Principle

Fast recall first. Precision and reasoning later.


In [3]:
# =========================================================
# JN TEST CELL — corpus_builder engine (should reproduce same results)
# Code lives at: /home/hello/Projects/Statements/code/adapter/corpus_builder.py
# =========================================================

import sys
import re
from pathlib import Path

# 1) Make sure Python can import your adapter module
ADAPTER_DIR = Path("/home/hello/Projects/Statements/code/adapter").resolve()
if str(ADAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(ADAPTER_DIR))

from corpus_builder import run_corpus_builder  # <- from /adapter/corpus_builder.py

# 2) Paths (same as your original cell)
INPUT_ROOT = Path(r"/media/hello/Vault/Tribunals/ET_Cases/").resolve()
MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()

# 3) Needles (calibration lives here)
NEEDLES_ALL = [
    "unfair dismissal",
]

NEEDLES_ANY = [
    "upheld",
    "verbal warning",
    "no contemporaneous evidence",
    "predetermination",
]

# 4) Regex (calibration lives here)
APPEAL_SCOPE_REGEX = re.compile(
    r"""
    (
        (not\s+raised\s+(?:in|within)\s+the\s+(?:written\s+)?appeal) |
        (outside\s+the\s+scope\s+of\s+the\s+appeal) |
        (declined\s+to\s+consider) |
        (refused\s+to\s+consider) |
        (limited\s+to\s+the\s+grounds) |
        (confined\s+to\s+(?:the\s+)?grounds) |
        (new\s+grounds\s+(?:raised|introduced)\s+(?:at|during)\s+the\s+appeal) |
        (raised\s+at\s+the\s+appeal\s+hearing) |
        (fresh\s+consideration) |
        (rubber\s+stamp) |
        (failed\s+to\s+engage\s+with) |
        (did\s+not\s+address\s+the\s+appeal\s+ground)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

ASSUMED_INTENTION_REGEX = re.compile(
    r"""
    (
        (assum(?:e|ed|ing)\s+(?:that\s+)?(?:the\s+)?(?:claimant|employee|appellant|respondent|he|she|they|you)\s+(?:had\s+)?(?:an?\s+)?intention) |
        (assum(?:e|ed|ing)\s+(?:the\s+)?(?:claimant|employee|appellant|respondent|he|she|they|you)\s+(?:was|were)\s+(?:intend(?:ing)?|trying)\s+to) |

        ((?:his|her|their|the)\s+intention\s+(?:was|had\s+been)\s+to) |
        (intend(?:ed|ing)?\s+to\s+(?:avoid|evade|get\s+out\s+of|circumvent)) |

        (motive\s+(?:was|had\s+been)\s+to) |
        (ulterior\s+motive) |
        (improper\s+motive) |

        (purpose\s+(?:was|had\s+been)\s+to) |
        (designed\s+to\s+(?:avoid|evade|circumvent)) |
        (with\s+the\s+(?:aim|objective|intention)\s+of) |

        (pretext) |
        (a\s+sham) |
        (smokescreen) |
        (cover\s+(?:story|for)) |

        (it\s+(?:can|could|may|might)\s+be\s+inferred\s+that) |
        (i\s+infer\s+that) |
        (the\s+tribunal\s+infers\s+that)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

# 5) Run (these overrides mirror your original defaults)
df = run_corpus_builder(
    input_root=INPUT_ROOT,
    matches_root=MATCHES_ROOT,
    needles_all=NEEDLES_ALL,
    needles_any=NEEDLES_ANY,
    appeal_scope_regex=APPEAL_SCOPE_REGEX,
    assumed_intention_regex=ASSUMED_INTENTION_REGEX,
    cfg_overrides=dict(
        # match the original defaults exactly
        case_sensitive=False,
        min_pages=4,
        text_pages_head=12,
        text_pages_tail=6,
        max_workers=24,
        submit_chunk_size=2000,
        preserve_structure=True,
        master_csv_name="_matches_index.csv",
        regex_appeal_folder_name="_APPEAL_SCOPE_REGEX",
        regex_intent_folder_name="_ASSUMED_INTENTION_REGEX",
        regex_only_folder_name="_REGEX_ONLY",
    ),
)

# 6) Quick sanity checks
print("\n[done] df shape:", df.shape)
print("[done] head:")
display(df.head(10))

# if you want an immediate “counts” view similar to your printouts:
if not df.empty:
    ok = df.get("ok", False).fillna(False).astype(bool)
    err = df.get("error", False).fillna(False).astype(bool)
    print("\n[counts]")
    print("ok:", int(ok.sum()), "| error:", int(err.sum()), "| total rows kept:", len(df))

    # how many matched *anything* (substring any OR regex)
    if "has__any_needle" in df.columns:
        print("has__any_needle:", int(df["has__any_needle"].fillna(False).astype(bool).sum()))

[scan] Input root: /media/hello/Vault/Tribunals/ET_Cases
[scan] PDFs found: 127755
[scan] NEEDLES_ALL (must match all): ['unfair dismissal']
[scan] NEEDLES_ANY (substring OR regex): ['upheld', 'verbal warning', 'no contemporaneous evidence', 'predetermination']
[scan] Pages >= 4
[scan] Text scan: head=12 pages, tail=6 pages
[scan] Workers=24 | submit_chunk=2000
[out] Principal matches folder: /media/hello/Vault/Tribunals/_Matches
[out] Preserve structure: True


Scanning PDFs: 100%|██████████| 127755/127755 [00:26<00:00, 4890.17file/s]


[stats] scanned=127755 | match=5062 | no_match=122692 | error=1
[stats] substring_any=3401 | appeal_scope_regex=261 | assumed_intention_regex=2322 | regex_only=1661
[scan] OK matches: 5062
[group] 'upheld' -> 3145 matches -> /media/hello/Vault/Tribunals/_Matches/upheld


Copying -> upheld: 100%|██████████| 3145/3145 [00:00<00:00, 4583.59file/s]


[group] 'verbal warning' -> 291 matches -> /media/hello/Vault/Tribunals/_Matches/verbal_warning


...

[regex] REGEX_ONLY (no substring NEEDLES_ANY) hits: 1661


Copying -> _REGEX_ONLY: 100%|██████████| 1661/1661 [00:00<00:00, 5911.90file/s]
/home/hello/Projects/Statements/code/adapter/corpus_builder.py:542: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["appeal_scope_hit"].fillna(False).astype(bool)
/home/hello/Projects/Statements/code/adapter/corpus_builder.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["assumed_intention_hit"].fillna(False).astype(bool)


[csv] Wrote single master CSV: /media/hello/Vault/Tribunals/_Matches/_matches_index.csv

[warn] Some PDFs failed parsing/opening. First 10 errors:
                                                                                                           path                                                                                                                                                           error_msg
/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Shaw_v_BUPA_Care_Homes__BNH__Ltd_2301363-16_and_2302838-16_Full.pdf EmptyFileError: Cannot open empty file: filename='/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Shaw_v_BUPA_Care_Homes__BNH__Ltd_2301363-16_and_2302838-16_Full.pdf'.

[done] df shape: (5063, 22)
[done] head:


,ok,error,path,pages,size_mb,mtime,hit_all,hit_any,appeal_scope_hit,appeal_scope_match,...,regex_only,error_msg,matches_root,has__upheld,has__verbal_warning,has__no_contemporaneous_evidence,has__predetermination,has__appeal_scope_regex,has__assumed_intention_regex,has__any_needle
0,True,False,/media/hello/Vault/Tribunals/ET_Cases/Ms_X_Ju_...,15.0,0.219,2026-01-16 15:58:49.281348944,unfair dismissal,,False,,...,True,NaN,/media/hello/Vault/Tribunals/_Matches,False,False,False,False,False,True,True
1,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mrs_S_Me...,36.0,0.901,2026-01-16 13:18:20.104724646,unfair dismissal,verbal warning,False,,...,False,NaN,/media/hello/Vault/Tribunals/_Matches,False,True,False,False,False,True,True
2,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_K_Mal...,8.0,0.147,2026-01-15 17:51:26.081697702,unfair dismissal,,False,,...,True,NaN,/media/hello/Vault/Tribunals/_Matches,False,False,False,False,False,True,True
3,True,False,/media/hello/Vault/Tribunals/ET_Cases/1600469....,9.0,0.114,2026-01-15 08:59:04.407406092,unfair dismissal,upheld,False,,...,False,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,False,False,False,False,True
4,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_P_Jon...,9.0,1.159,2026-01-16 06:25:54.110433578,unfair dismissal,,False,,...,True,NaN,/media/hello/Vault/Tribunals/_Matches,False,False,False,False,False,True,True
5,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_J_Leh...,32.0,0.279,2026-01-15 17:17:15.024190426,unfair dismissal,predetermination,False,,...,False,NaN,/media/hello/Vault/Tribunals/_Matches,False,False,False,True,False,False,True
6,True,False,/media/hello/Vault/Tribunals/ET_Cases/Miss_S_D...,37.0,0.486,2026-01-15 12:40:22.228126526,unfair dismissal,upheld,False,,...,False,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,False,False,False,False,True
7,True,False,/media/hello/Vault/Tribunals/ET_Cases/Sally_Cl...,5.0,0.146,2026-01-16 17:28:54.397037506,unfair dismissal,upheld,False,,...,False,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,False,False,False,False,True
8,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_S_Rad...,22.0,0.294,2026-01-16 07:50:58.618892431,unfair dismissal,,True,failed to engage with,...,True,NaN,/media/hello/Vault/Tribunals/_Matches,False,False,False,False,True,False,True
9,True,False,/media/hello/Vault/Tribunals/ET_Cases/Miss_V_W...,50.0,0.669,2026-01-15 12:57:52.583752871,unfair dismissal,upheld,False,,...,False,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,False,False,False,True,True



[counts]
ok: 5062 | error: 1 | total rows kept: 5063
has__any_needle: 5062


In [2]:
# =========================================================
# JN TEST CELL — corpus_builder engine (should reproduce same results)
# Code lives at: /home/hello/Projects/Statements/code/adapter/corpus_builder.py
# =========================================================

import sys
import re
import pandas as pd
from pathlib import Path

# 1) Make sure Python can import your adapter module
ADAPTER_DIR = Path("/home/hello/Projects/Statements/code/adapter").resolve()
if str(ADAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(ADAPTER_DIR))

from corpus_builder import run_corpus_builder

# =========================================================
# Paths
# =========================================================

INPUT_ROOT = Path("/media/hello/Vault/Tribunals/ET_Cases/").resolve()
MATCHES_ROOT = Path("/media/hello/Vault/Tribunals/_Matches").resolve()

# =========================================================
# Load regex patterns from CSV (fully automated)
# =========================================================

PATTERN_CSV = Path(
    "/home/hello/Projects/Statements/output/needle_regex_candidates.csv"
).resolve()

patterns_df = pd.read_csv(PATTERN_CSV)

patterns_df = patterns_df.dropna(subset=["regex_pattern"])

CSV_REGEX_PATTERNS = patterns_df["regex_pattern"].tolist()

print("Loaded regex patterns:", len(CSV_REGEX_PATTERNS))

# =========================================================
# Clean regex flags so Python can compile them together
# =========================================================

CLEAN_REGEX_PATTERNS = [
    re.sub(r"\(\?[ixms]+\)", "", p) for p in CSV_REGEX_PATTERNS
]

combined_regex = re.compile(
    "|".join(CLEAN_REGEX_PATTERNS),
    re.IGNORECASE | re.VERBOSE
)

# =========================================================
# Needles
# =========================================================

NEEDLES_ALL = [
    "unfair dismissal",
]

# disable substring matching
NEEDLES_ANY = []

# =========================================================
# Run corpus builder
# =========================================================

df = run_corpus_builder(
    input_root=INPUT_ROOT,
    matches_root=MATCHES_ROOT,
    needles_all=NEEDLES_ALL,
    needles_any=NEEDLES_ANY,
    appeal_scope_regex=combined_regex,
    assumed_intention_regex=re.compile(r"$^"),  # disabled
    cfg_overrides=dict(
        case_sensitive=False,
        min_pages=4,
        text_pages_head=12,
        text_pages_tail=6,
        max_workers=24,
        submit_chunk_size=2000,
        preserve_structure=True,
        master_csv_name="_matches_index.csv",
        regex_appeal_folder_name="_APPEAL_SCOPE_REGEX",
        regex_intent_folder_name="_ASSUMED_INTENTION_REGEX",
        regex_only_folder_name="_REGEX_ONLY",
    ),
)

# =========================================================
# Quick sanity checks
# =========================================================

print("\n[done] df shape:", df.shape)
print("[done] head:")
display(df.head(10))

if not df.empty:
    ok = df.get("ok", False).fillna(False).astype(bool)
    err = df.get("error", False).fillna(False).astype(bool)

    print("\n[counts]")
    print("ok:", int(ok.sum()), "| error:", int(err.sum()), "| total rows kept:", len(df))

    if "has__any_needle" in df.columns:
        print(
            "has__any_needle:",
            int(df["has__any_needle"].fillna(False).astype(bool).sum())
        )

Loaded regex patterns: 6
[scan] Input root: /media/hello/Vault/Tribunals/ET_Cases
[scan] PDFs found: 127755
[scan] NEEDLES_ALL (must match all): ['unfair dismissal']
[scan] NEEDLES_ANY (substring OR regex): []
[scan] Pages >= 4
[scan] Text scan: head=12 pages, tail=6 pages
[scan] Workers=24 | submit_chunk=2000
[out] Principal matches folder: /media/hello/Vault/Tribunals/_Matches
[out] Preserve structure: True


Scanning PDFs: 100%|██████████| 127755/127755 [00:20<00:00, 6385.66file/s]


[stats] scanned=127755 | match=49 | no_match=127705 | error=1
[stats] substring_any=0 | appeal_scope_regex=49 | assumed_intention_regex=0 | regex_only=49
[scan] OK matches: 49
[regex] APPEAL_SCOPE_REGEX hits: 49


Copying -> _APPEAL_SCOPE_REGEX: 100%|██████████| 49/49 [00:00<00:00, 1369.45file/s]


[regex] ASSUMED_INTENTION_REGEX hits: 0
[regex] REGEX_ONLY (no substring NEEDLES_ANY) hits: 49


Copying -> _REGEX_ONLY: 100%|██████████| 49/49 [00:00<00:00, 4607.27file/s]
/home/hello/Projects/Statements/code/adapter/corpus_builder.py:542: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["appeal_scope_hit"].fillna(False).astype(bool)
/home/hello/Projects/Statements/code/adapter/corpus_builder.py:548: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out["assumed_intention_hit"].fillna(False).astype(bool)


[csv] Wrote single master CSV: /media/hello/Vault/Tribunals/_Matches/_matches_index.csv

[warn] Some PDFs failed parsing/opening. First 10 errors:
                                                                                                           path                                                                                                                                                           error_msg
/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Shaw_v_BUPA_Care_Homes__BNH__Ltd_2301363-16_and_2302838-16_Full.pdf EmptyFileError: Cannot open empty file: filename='/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Shaw_v_BUPA_Care_Homes__BNH__Ltd_2301363-16_and_2302838-16_Full.pdf'.

[done] df shape: (50, 18)
[done] head:


,ok,error,path,pages,size_mb,mtime,hit_all,hit_any,appeal_scope_hit,appeal_scope_match,assumed_intention_hit,assumed_intention_match,regex_only,error_msg,matches_root,has__appeal_scope_regex,has__assumed_intention_regex,has__any_needle
0,True,False,/media/hello/Vault/Tribunals/ET_Cases/Professo...,68.0,0.885,2026-01-16 16:09:36.699881554,unfair dismissal,,True,disproportionate sanction,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
1,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_D_Dip...,30.0,0.280,2026-01-15 14:48:26.400677919,unfair dismissal,,True,"not told the truth, putting patients potential...",False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
2,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_C_Ber...,29.0,0.431,2026-01-15 14:15:34.209166288,unfair dismissal,,True,disproportionate sanction,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
3,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Lo...,21.0,0.169,2026-01-16 12:29:23.136904478,unfair dismissal,,True,not told the Claimant that she might be at risk,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
4,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_I_M_D...,21.0,0.319,2026-01-15 16:45:55.837406874,unfair dismissal,,True,Disproportionate sanction,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
5,True,False,/media/hello/Vault/Tribunals/ET_Cases/Miss_S_Y...,9.0,0.312,2026-01-15 12:50:26.635379314,unfair dismissal,,True,not informed her role was at risk,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
6,False,True,/media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Sh...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EmptyFileError: Cannot open empty file: filena...,/media/hello/Vault/Tribunals/_Matches,False,False,False
7,True,False,/media/hello/Vault/Tribunals/ET_Cases/2423941-...,10.0,0.244,2026-01-15 09:10:52.138251781,unfair dismissal,,True,"no warning, consultation, at risk",False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
8,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mrs_D_Ta...,12.0,0.073,2026-01-16 09:18:24.451442480,unfair dismissal,,True,inconsistent activities,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True
9,True,False,/media/hello/Vault/Tribunals/ET_Cases/Mr_D_Neg...,9.0,0.176,2026-01-15 15:04:02.338757753,unfair dismissal,,True,disproportionate \nsanction,False,,True,NaN,/media/hello/Vault/Tribunals/_Matches,True,False,True



[counts]
ok: 49 | error: 1 | total rows kept: 50
has__any_needle: 49


In [ ]:
df = pd.read_csv("/home/hello/Projects/Statements/output/needle_regex_candidates.csv")

print(df.columns)
print(df.head(10))

print(df.shape)

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================

MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()
MASTER_CSV_NAME = "_matches_index.csv"

# =========================================================
# LOAD
# =========================================================

csv_path = MATCHES_ROOT / MASTER_CSV_NAME

df = pd.read_csv(csv_path)

print(f"[info] Total matched cases: {len(df)}")
print(f"[info] Columns: {len(df.columns)}")

# =========================================================
# FIND BOOLEAN MATCH COLUMNS
# =========================================================

match_cols = [c for c in df.columns if c.startswith("has__")]

if not match_cols:
    print("No has__ columns found.")
    raise SystemExit

# Ensure boolean
for c in match_cols:
    df[c] = df[c].fillna(False).astype(bool)

# =========================================================
# FREQUENCY CALCULATION
# =========================================================

freq_rows = []

total = len(df)

for col in match_cols:
    count = df[col].sum()
    pct = round((count / total) * 100, 2) if total > 0 else 0
    freq_rows.append({
        "match_type": col,
        "count": int(count),
        "percentage_of_total": pct
    })

freq_df = pd.DataFrame(freq_rows).sort_values(
    by="count",
    ascending=False
).reset_index(drop=True)

print("\n=== MATCH FREQUENCY TABLE ===\n")
print(freq_df)

# Optional: save
freq_out = MATCHES_ROOT / "_match_frequencies.csv"
freq_df.to_csv(freq_out, index=False)
print(f"\n[csv] Frequency table written to: {freq_out}")

# MOLTIE – Jupyter Inference Cell (y_spec + Needle Runner)

## Purpose

This notebook cell executes the `y_spec.py` schema inference module **and** a closed-set needle classifier over each row of a witness statement CSV file. It is designed for **interactive debugging, enrichment, and calibration**, not bulk production execution.

It enables row-level inspection of semantic tagging (needles), structured Y inference, JSON validity, and stability before transitioning to a production batch workflow.

---

## Input

* **CSV file**:
  `/home/hello/Projects/Statements/input/Leonardo_WS.csv`

* **Column used**:
  `text_verbatim`

Each row from `text_verbatim` is treated as `ws_text` and:

1. Passed via `stdin` to `y_spec.py`
2. Independently evaluated by the needle classifier (LLM closed-set tagging)

---

## Outputs

Two files are written to:

`/home/hello/Projects/Statements/output`

1. **Leonardo_WS_enhanced.csv**

   * Contains **all original CSV columns**
   * Adds:

     * `needle_selected_raw`
     * `has__/conf__/quote__` columns per tag
     * `y_ok`, `y_rc`
     * `ws_len`
     * `X1` (1-based row id)
     * `doc`

2. **Y_inferred.json**

   * Aggregates structured `y_spec` output per processed row
   * Preserves full Y JSON for auditability

---

## Execution Flow

### 1. Configuration

The cell defines:

* Model name (`mistral-small3.2:latest`)
* Path to `y_spec.py`
* Debug mode toggle
* Flexible row selector (`DEBUG_SLICE`)
* Needle tag taxonomy (closed set)

Debug selector supports:

* `"3"` → process exactly the 3rd row (1-based)
* `":5"` → first 5 rows
* `"2:5"` → Python slice semantics

This allows precise surgical debugging.

---

### 2. CSV Loading

* Reads the CSV using pandas
* Validates `text_verbatim` exists
* Converts nulls to empty strings
* Determines which rows to process (full run or debug slice)

Original CSV structure is preserved.

---

### 3. Row Iteration (with tqdm)

For each selected row:

* Assigns a deterministic 1-based identifier (`X1`)
* Strips whitespace
* Skips empty rows
* Prints trace:

```
X1=<row_number> doc=Leonardo_WS.csv
```

This guarantees reproducibility and traceability.

---

### 4. Dual Processing Per Row

Each `ws_text` flows through two independent pipes:

#### A) Needle Classifier (Semantic Tagging)

* Closed-set LLM classification
* Evidence-quoted
* Negation-aware
* JSON-validated
* Returns:

  * Selected tags
  * Confidence scores
  * Evidence quotes

Expanded into structured columns (`has__/conf__/quote__`).

#### B) Y-Spec Structured Inference

```
python y_spec.py --model mistral-small3.2:latest
```

* Input via `stdin`
* Captures:

  * `stdout`
  * `stderr`
  * `returncode`
* Attempts immediate JSON parsing
* Aggregates valid results into `Y_inferred.json`

Needle tagging and Y inference remain structurally independent.

---

### 5. Enrichment Merge

The enrichment fields are merged back into the **full original DataFrame**, ensuring:

* No original data is lost
* Only processed rows receive populated enrichment fields (in debug mode)
* Non-processed rows remain intact

---

## Why This Design Is Intentional

This notebook version prioritises:

* Structural independence between retrieval (needles) and reasoning (Y)
* Full preservation of source data
* Evidence-anchored tagging
* Deterministic debug slicing
* Immediate JSON validation
* Transparent failure surfaces

It does **not** parallelise.

It does **not** bias X/Y extraction toward needle concepts.

It maintains clean architectural separation between semantic tagging and structured inference.

---

## When to Transition to Production Script

Move to a full `.py` batch runner once:

* Needle classification is stable
* Y JSON schema is consistent
* Debug slicing no longer required
* Error patterns are understood

Production version should then:

* Support parallel execution
* Implement retry logic
* Log failures explicitly
* Optionally write per-row Y JSON files

---

## Role in the MOLTIE Architecture

This cell functions as a **calibration and enrichment harness**.

It sits between:

* Raw witness statement substrate
* Independent semantic tagging layer
* Structured Y inference layer

It ensures that:

* Retrieval signals (needles)
* Structural reasoning signals (Y)

are derived independently from the same text.

---

## Summary

This Jupyter cell provides a controlled inference environment that:

* Reads witness statement rows
* Applies independent needle classification
* Executes `y_spec.py`
* Validates structured JSON output
* Merges enrichment into the original dataset
* Writes consolidated outputs

It is modular, auditable, deterministic under debug, and structurally clean.


In [ ]:
import json, re
from pathlib import Path

def canonicalize_tag(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def build_tag_spec(
    needles_any,
    needles_all,
    regex_map,          # dict: {"appeal_scope_regex": compiled_regex, ...}
    matches_root: Path,
    df=None,            # optional corpus df to attach prevalence
    out_name="_tag_spec.json",
):
    tags = []

    # --- NEEDLES_ANY become model-selectable tags ---
    for phrase in needles_any:
        tag = canonicalize_tag(phrase)
        tags.append({
            "tag": tag,
            "type": "needle_any",
            "source": {"phrase": phrase},
            "desc": f"keyword/phrase match: {phrase!r}",
        })

    # --- Regex tags are explicitly named in regex_map keys ---
    for tag_name, rx in regex_map.items():
        tag = canonicalize_tag(tag_name)
        tags.append({
            "tag": tag,
            "type": "regex",
            "source": {"name": tag_name},
            "desc": f"regex match: {tag_name}",
        })

    # --- (Optional) include NEEDLES_ALL as a non-model “gate” tag ---
    # You usually do NOT want the model to output this; it’s a corpus filter.
    needles_all_norm = [canonicalize_tag(x) for x in needles_all]

    # Deduplicate tags by "tag"
    seen = set()
    deduped = []
    for t in tags:
        if t["tag"] in seen:
            continue
        seen.add(t["tag"])
        deduped.append(t)
    tags = deduped

    # Attach prevalence if df has `has__{tag}` columns
    if df is not None and len(df) > 0:
        n = len(df)
        cols = set(df.columns)

        for t in tags:
            col = f"has__{t['tag']}"
            t["corpus_column"] = col
            if col in cols:
                s = df[col].fillna(False)
                try:
                    b = s.astype(bool)
                except Exception:
                    b = s.apply(bool)
                cnt = int(b.sum())
                pct = (cnt / n) * 100.0
                t["stats"] = {"count": cnt, "pct": pct, "n_cases": n}
                t["desc"] = f"{t['desc']} (hits {cnt}/{n}; {pct:.2f}%)"
            else:
                t["stats"] = {"count": None, "pct": None, "n_cases": n}
                t["desc"] = f"{t['desc']} (missing corpus column {col!r})"

    spec = {
        "version": "v1",
        "needles_all_gate": needles_all_norm,   # used for corpus filtering, not model output
        "computed": ["any_needle", "none"],     # reserved/system tags
        "tags": tags,
    }

    out_path = matches_root / out_name
    out_path.write_text(json.dumps(spec, indent=2, ensure_ascii=False), encoding="utf-8")
    return out_path

# ✅ your exact setup
REGEX_MAP = {
    "appeal_scope_regex": APPEAL_SCOPE_REGEX,
    "assumed_intention_regex": ASSUMED_INTENTION_REGEX,
}

# If you already have df from run_corpus_builder, pass df=df to add prevalence
tag_spec_path = build_tag_spec(
    needles_any=NEEDLES_ANY,
    needles_all=NEEDLES_ALL,
    regex_map=REGEX_MAP,
    matches_root=MATCHES_ROOT,
    df=df,  # optional; safe if df exists
    out_name="_tag_spec.json",
)

print("[tag_spec] written:", tag_spec_path)

In [ ]:
import json
from pathlib import Path

def load_tag_spec(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def build_vocab_from_tag_spec(spec: dict):
    tags = [t["tag"] for t in spec["tags"]]
    # preserve order, unique
    seen = set()
    tags_u = []
    for t in tags:
        if t in seen:
            continue
        seen.add(t)
        tags_u.append(t)

    allowed = []
    if "any_needle" in spec.get("computed", []):
        allowed.append("any_needle")
    allowed.extend(tags_u)
    if "none" in spec.get("computed", []):
        allowed.append("none")

    defs = {t["tag"]: (t.get("desc") or f"tag: {t['tag']}") for t in spec["tags"]}
    if "any_needle" in spec.get("computed", []):
        defs["any_needle"] = "computed: at least one non-none tag applies (deterministic; not model-driven)"
    if "none" in spec.get("computed", []):
        defs["none"] = "none of the above apply"

    return allowed, defs

TAG_SPEC_PATH = Path("/media/hello/Vault/Tribunals/_Matches/_tag_spec.json").resolve()
spec = load_tag_spec(TAG_SPEC_PATH)

ALLOWED_TAGS, TAG_DEFS = build_vocab_from_tag_spec(spec)

print("[vocab] loaded from:", TAG_SPEC_PATH)
print("ALLOWED_TAGS:", ALLOWED_TAGS)

In [ ]:
import sys, re, json
from pathlib import Path
import pandas as pd

# import engine
ADAPTER_DIR = Path("/home/hello/Projects/Statements/code/adapter").resolve()
if str(ADAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(ADAPTER_DIR))

from y_runner_engine import YRunnerConfig, run_y_pipeline, canonicalize_tag

# =========================================================
# CALIBRATION INPUTS (OWNED BY JN)
# =========================================================
CSV_PATH = Path("/home/hello/Projects/Statements/input/Leonardo_WS.csv")
TEXT_COL = "text_verbatim"

OUT_DIR = Path("/home/hello/Projects/Statements/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ENHANCED_CSV = OUT_DIR / "Leonardo_WS_enhanced.csv"
OUT_Y_JSON = OUT_DIR / "Y_inferred.json"

Y_SPEC_PY = Path("/home/hello/Projects/Statements/code/moltie/schemas/y_spec.py")
MODEL = "mistral-small3.2:latest"

DEBUG = False
DEBUG_SLICE = "3"

NEEDLE_TIMEOUT = 180
Y_SPEC_TIMEOUT = 180

# =========================================================
# AUTO-VOCAB (FROM CORPUS BUILDER TAGSPEC)
# =========================================================
MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()
TAG_SPEC_PATH = (MATCHES_ROOT / "_tag_spec.json").resolve()

def load_tag_spec(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"TagSpec not found at: {path}\n"
            f"Run the corpus_builder cell first to generate {_tag_spec.json!r}."
        )
    return json.loads(path.read_text(encoding="utf-8"))

def build_allowed_tags_and_defs_from_spec(spec: dict):
    if "tags" not in spec or not isinstance(spec["tags"], list):
        raise ValueError("Invalid TagSpec: missing 'tags' list")

    # Preserve order, unique
    tag_list = []
    seen = set()
    for t in spec["tags"]:
        tag = t.get("tag")
        if not tag or not isinstance(tag, str):
            continue
        if tag in seen:
            continue
        seen.add(tag)
        tag_list.append(tag)

    computed = spec.get("computed", [])
    allowed = []
    if "any_needle" in computed:
        allowed.append("any_needle")
    allowed.extend(tag_list)
    if "none" in computed:
        allowed.append("none")

    defs = {}
    for t in spec["tags"]:
        tag = t.get("tag")
        if not tag:
            continue
        desc = (t.get("desc") or "").strip()
        defs[tag] = desc if desc else f"tag: {tag}"

    # Stable defs for computed/system tags
    if "any_needle" in computed:
        defs["any_needle"] = "computed: at least one non-none tag applies (deterministic; not model-driven)"
    if "none" in computed:
        defs["none"] = "none of the above apply"

    return allowed, defs

spec = load_tag_spec(TAG_SPEC_PATH)
ALLOWED_TAGS, TAG_DEFS = build_allowed_tags_and_defs_from_spec(spec)

print("[tag_spec] loaded:", TAG_SPEC_PATH)
print("[tag_spec] allowed tags:", ALLOWED_TAGS)

# =========================================================
# PROMPT BUILDER (now driven by TagSpec)
# =========================================================
def make_needle_prompt(text: str) -> str:
    # model should NOT output "any_needle"
    model_allowed = [t for t in ALLOWED_TAGS if t not in ("any_needle",)]
    tag_lines = "\n".join([f'- "{t}": {TAG_DEFS.get(t, "")}' for t in model_allowed])

    return f"""
TASK:
Given TEXT, select all applicable TAGS from the allowed list.

CRITICAL TAG RULE:
- Each "tag" value MUST be EXACTLY one of the strings in ALLOWED_TAGS (character-for-character).
- Do NOT invent new tag names.
- Do NOT output "any_needle" (it will be computed downstream).
- If uncertain, return ONLY ["none"].

ALLOWED_TAGS:
{tag_lines}

RULES:
- For every selected tag (except "none"), provide:
  - confidence in [0,1]
  - evidence_quote copied verbatim from TEXT (max 200 chars)
  - negated: true if TEXT explicitly indicates the opposite
- If you cannot quote evidence from TEXT, do NOT select that tag.
- Return VALID JSON ONLY. No markdown, no commentary, no extra keys.

TEXT:
<<<
{text.strip()}
>>>

OUTPUT JSON SCHEMA:
{{
  "selected": [
    {{"tag": "...", "confidence": 0.0, "negated": false, "evidence_quote": "..."}}
  ]
}}
""".strip()

# =========================================================
# ENGINE CONFIG (paths/timeouts are inputs too)
# =========================================================
cfg = YRunnerConfig(
    csv_path=CSV_PATH,
    text_col=TEXT_COL,
    out_dir=OUT_DIR,
    out_enhanced_csv=OUT_ENHANCED_CSV,
    out_y_json=OUT_Y_JSON,
    y_spec_py=Y_SPEC_PY,
    model=MODEL,
    debug=DEBUG,
    debug_slice=DEBUG_SLICE,
    needle_timeout=NEEDLE_TIMEOUT,
    y_spec_timeout=Y_SPEC_TIMEOUT,
    matches_root=MATCHES_ROOT,
    # IMPORTANT: this should point to a case index / tag-spec aware source,
    # but leaving as-is for now per your current engine behavior.
    master_csv_name="_match_frequencies.csv",
    strict_schema_gate=True,
)

df_out, y_results, diag = run_y_pipeline(
    cfg=cfg,
    allowed_tags=ALLOWED_TAGS,
    tag_defs=TAG_DEFS,
    needle_prompt_fn=make_needle_prompt,
    canonicalize_fn=canonicalize_tag,
)

display(df_out.head(25))
if diag is not None:
    display(diag.head(50))